If GLUE package not published, use the uploaded result table to generate plots:
results/GLUE/glomerular_gcmc_withshuffle_24bg_high_low_scale_0.0_10000points.csv

In [ ]:
import numpy as np
from odor_mix_code.coding_model_fanofactor import *
from odor_mix_code.paths import PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, ensure_output_dirs
ensure_output_dirs()
from gcmc import gcmc_analysis
from gcmc.simulation_analysis import simulation_capacity
from gcmc.preprocess import downsample_manifolds, reallocate_origin, shuffle_manifolds
import time
from matplotlib.lines import Line2D
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from joblib import Parallel, delayed
import numpy as np
import time
import pandas as pd
from itertools import product

## generate responses and compute geometry

In [ ]:
# main run + shuffled run (~8 hours on macbook M4 pro) - FANO FACTOR RUN
sigmas = [0.0]
cue_concs = [1e-2, 1.0, 10**(-4.5)]
cue_concs = [10**(-4.5)]
fano_factors = [0.01, 0.0, 0.5, 0.8, 1.0, 1.2, 1.5]
bg_concs = [3.2e-3, 1e-1]
train_cue_concs = np.arange(10)
total_bg = 24
scale = 0.0
n_points = 10000

def process_combo(fano_factor, bg_conc, train_cue_conc):
    """Generate data for this triple, loop over conc_idx & n_bg, run gcmc_analysis."""
    results_local = []
    # 1) generate all the train/test sets for this combination
    X_train, y_train, X_tests, y_tests, all_test_concs, all_target_concs = \
        generate_datasets(
            n_train=1,
            n_test=n_points,
            cue_concs=cue_concs,
            train_cue_conc=train_cue_conc,
            cue=-1,
            n_bg=-1,
            model_params=init_model_parameters(rho=0.5, bg_conc=bg_conc, total_bg=total_bg, seed=int(train_cue_conc), fano_factor=fano_factor),
                        fano_factor=fano_factor,
        )
    # 2) for each test‐concentration index and each number of backgrounds
    for conc_idx in range(len(cue_concs)):
        for n_bg in [0, 1, 2, 4, 8, 12, 16]:
            mask = np.sum(all_test_concs[conc_idx,:,2:] > 0, axis=-1) == n_bg
            X_test = X_tests[conc_idx][mask]
            y_test = y_tests[conc_idx][mask]
            mfld1 = X_test[y_test == 0].T
            mfld2 = X_test[y_test == 1].T
            manifolds = [mfld1, mfld2]
            dichotomies = [[-1, 1]]

            start = time.time()
            # manifolds = downsample_manifolds(manifolds, n_points=n_points)
            realloc_manifolds = reallocate_origin(manifolds, scale=scale) # RESCALE ORIGIN (add bias)

            try:
                ret = gcmc_analysis(realloc_manifolds, dichotomies)
                simcap = simulation_capacity(realloc_manifolds, dichotomies)
                elapsed = time.time() - start

                # shuffled capacity
                manifolds_shuffled = shuffle_manifolds(manifolds, seed=42)
                manifolds_reallocated = reallocate_origin(manifolds_shuffled, scale=scale)
                shuffle_ret = gcmc_analysis(manifolds_reallocated, dichotomies)


                # optional: print shapes & timing
                print(f"[fano factor={fano_factor}, bg={bg_conc}, train_cue={train_cue_conc}, "
                    f"conc={cue_concs[conc_idx]:.1e}, n_bg={n_bg}] "
                    f"→ cap={ret['capacity']:.2f}, sim_cap={simcap:.2f} in {elapsed:.1f}s")

                results_local.append([
                    fano_factor,
                    train_cue_conc,
                    bg_conc,
                    cue_concs[conc_idx],
                    n_bg,
                    ret['capacity'],
                    simcap,
                    ret['dimension'],
                    ret['radius'],
                    ret['utility'],
                    ret['center_alignment'],
                    ret['axis_alignment'],
                    ret['center_axis_alignment'],
                    shuffle_ret['capacity'],
                    shuffle_ret['dimension'],
                    shuffle_ret['radius'],
                    shuffle_ret['utility'],
                    shuffle_ret['center_alignment'],
                    shuffle_ret['axis_alignment'],
                    shuffle_ret['center_axis_alignment'],
                ])
            except Exception as e:
                print(f"FAILED: fano={fano_factor}, bg={bg_conc}, train={train_cue_conc}, "
                    f"conc={cue_concs[conc_idx]:.1e}, n_bg={n_bg}, err={e}")

                results_local.append([
                    fano_factor,              # fano_factor
                    train_cue_conc,           # train_cue_conc
                    bg_conc,                  # bg_conc
                    cue_concs[conc_idx],      # cue_conc
                    n_bg,                     # n_bg
                    None,                     # capacity
                    None,                     # capacity_simulation
                    None,                     # dimension
                    None,                     # radius
                    None,                     # utility
                    None,                     # center_alignment
                    None,                     # axis_alignment
                    None,                     # center_axis_alignment
                    None,                     # shuffle_capacity
                    None,                     # shuffle_dimension
                    None,                     # shuffle_radius
                    None,                     # shuffle_utility
                    None,                     # shuffle_center_alignment
                    None,                     # shuffle_axis_alignment
                    None,                     # shuffle_center_axis_alignment
                ])
    return results_local

# build list of all combinations
combos = list(product(fano_factors, bg_concs, train_cue_concs))
print(len(combos))

# run in parallel
all_results = Parallel(
    n_jobs=min(12, len(combos)),         # use all available cores
    verbose=10,        # a little progress info
    backend='loky'     # default safe backend
)(
    delayed(process_combo)(fano_factor, bg_conc, train_cue_conc)
    for fano_factor, bg_conc, train_cue_conc in combos
)

# flatten list-of-lists
results = [row for sublist in all_results for row in sublist]

columns = [
    'fano_factor',
    'train_cue_conc',
    'bg_conc',
    'cue_conc',
    'n_bg',
    'capacity',
    'capacity_simulation',
    'dimension',
    'radius',
    'utility',
    'center_alignment',
    'axis_alignment',
    'center_axis_alignment',
    'shuffle_capacity',
    'shuffle_dimension',
    'shuffle_radius',
    'shuffle_utility',
    'shuffle_center_alignment',
    'shuffle_axis_alignment',
    'shuffle_center_axis_alignment'
]

# Create DataFrame
df = pd.DataFrame(results, columns=columns)

# (optional) inspect
print(df.head())

# Save to CSV (no index column)
df.to_csv(f'{RESULTS_DIR}/GLUE/glomerular_poisson_weak_target_gcmc_withshuffle_{total_bg}bg_high_low_scale_{scale}_{n_points}points.csv', index=False)

## Plot Fig 4 from pre-computed results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
# import scienceplots
plt.style.use('default') #['science', 'no-latex'])
total_bg = 24
scale = 0.0
n_points = 10000
df = pd.read_csv(f'{RESULTS_DIR}/GLUE/glomerular_poisson_gcmc_withshuffle_{total_bg}bg_high_low_scale_{scale}_{n_points}points.csv')
df['Background'] = df['bg_conc']
df['Background odors'] = df['n_bg']
df['Cue'] = df['cue_conc']

# 1) define exactly how each n_bg level should look
palette = {
    0:  'black',    # n_bg==0
    16:  '#deebf7',  # lightest blue
    12:  '#c6dbef',
    8:  '#9ecae1',
    4:  '#6baed6',
    2: '#3182bd',
    1: '#08519c',  # darkest blue
}
palette = {
    0:  'black',    # n_bg==0
    1:  '#deebf7',  # lightest blue
    2:  '#c6dbef',
    4:  '#9ecae1',
    8:  '#6baed6',
    12: '#3182bd',
    16: '#08519c',  # darkest blue
}

palette = {
    0:  'black',
    1:  '#ebb08c',
    2:  "#dc7756",
    4:  '#b9324e',
    8:  '#9b2857',
    12:  '#682946',
    16:  '#3e0f2b',
}


dashes = {
    0:  (3, 1),  # dashed for n_bg==0
    1:  '',      # solid for all others
    2:  '',
    4:  '',
    8:  '',
    12: '',
    16: '',
}

In [ ]:
df['capacity_ratio_shuffle'] = df.capacity / df.shuffle_capacity
df = df[df.fano_factor != 0.0]
sub_df = df[(df['bg_conc'] == 3.2e-3) | (df['bg_conc'] == 1e-1)]
sub_df = sub_df[np.logical_or(sub_df['cue_conc'] == 1e-2, sub_df['cue_conc'] == 1.0)]
sub_df


# create a combined facet column, e.g. "Cue=1.0, BG=1e-4"
sub_df = sub_df.copy()
sub_df['panel_label'] = (
    'Target conc.=' + sub_df['cue_conc'].astype(str)
    + ',\nBackground conc.=' + sub_df['Background'].astype(str)
)


In [ ]:
# ----------------------------
# prepare dataframe
# ----------------------------
df = df.copy()
df["capacity_ratio_shuffle"] = df["capacity"] / df["shuffle_capacity"]

# keep just 2 cue concentrations and 2 background concentrations
sub_df = df[
    df["cue_conc"].isin([1e-2, 1.0]) &
    df["bg_conc"].isin([3.2e-3, 0.1])
].copy()

# make a clean panel label
sub_df["panel_label"] = (
    "Target conc.=" + sub_df["cue_conc"].astype(str) +
    ",\nBackground conc.=" + sub_df["bg_conc"].astype(str)
)

# choose panel order explicitly so columns appear in a predictable order
panel_order = [
    "Target conc.=0.01,\nBackground conc.=0.0032",
    "Target conc.=0.01,\nBackground conc.=0.1",
    "Target conc.=1.0,\nBackground conc.=0.0032",
    "Target conc.=1.0,\nBackground conc.=0.1",
]

# if your string formatting comes out slightly differently, generate it instead:
# panel_order = [
#     f"Target conc.={cue},\nBackground conc.={bg}"
#     for cue in [1e-2, 1.0]
#     for bg in [3.2e-3, 1.0]
# ]

metrics = [
    ("capacity", "Manifold capacity", True),
    ("dimension", "Effective dimension", True),
    ("radius", "Effective radius", False),
]

sns.set_context("paper", font_scale=1.7)

fig, axes = plt.subplots(
    nrows=3, ncols=4,
    figsize=(16, 10),
    dpi=600,
    sharex=True,
    sharey='row',
    constrained_layout=True
)

for col_idx, panel_label in enumerate(panel_order):
    panel_df = sub_df[sub_df["panel_label"] == panel_label]

    for row_idx, (metric, ylab, use_logy) in enumerate(metrics):
        ax = axes[row_idx, col_idx]

        sns.lineplot(
            data=panel_df,
            x="fano_factor",
            y=metric,
            hue="Background odors",
            style="Background odors",
            dashes=dashes,
            palette=palette,
            linewidth=3.0,
            ax=ax,
            legend=(row_idx == 0 and col_idx == 0),  # only make one legend
        )

        if use_logy:
            ax.set_yscale("log")

        # titles only on top row
        if row_idx == 0:
            ax.set_title(panel_label)

        # y-labels only on left column
        if col_idx == 0:
            ax.set_ylabel(ylab)
        else:
            ax.set_ylabel("")

        # x-labels only on bottom row
        if row_idx == len(metrics) - 1:
            ax.set_xlabel("Noise Fano Factor")
        else:
            ax.set_xlabel("")

        # optional per-row limits
        # if metric == "capacity":
        #     ax.set_ylim(0.02, 4.0)
        # if metric == "dimension":
        #     ax.set_ylim(...)
        # if metric == "radius":
        #     ax.set_ylim(...)

        # remove duplicate legends everywhere except first axis
        if not (row_idx == 0 and col_idx == 0):
            leg = ax.get_legend()
            if leg is not None:
                leg.remove()

for ax in axes[:, 3]:
    fig.delaxes(ax)
fig.set_constrained_layout_pads(w_pad=0.02, h_pad=0.02, wspace=0.1, hspace=0.1)
# move single legend outside
handles, labels = axes[0, 0].get_legend_handles_labels()
if axes[0, 0].get_legend() is not None:
    axes[0, 0].get_legend().remove()

fig.legend(
    handles, labels,
    title="Background #",
    loc="center left",
    bbox_to_anchor=(0.77, 0.5),
    frameon=False
)

plt.savefig(
    f"{FIGURES_DIR}/poisson_capacity_dimension_radius_bg_neural_noise_3x4_scale_{scale}_{n_points}points.tiff",
    dpi=600,
    bbox_inches="tight"
)
plt.savefig(
    f"{FIGURES_DIR}/poisson_capacity_dimension_radius_bg_neural_noise_3x4_scale_{scale}_{n_points}points.svg",
    dpi=600,
    bbox_inches="tight"
)
sns.despine()
plt.show()